# Enrichissement d'une offre Indeed

In [10]:
# À exécuter uniquement si SeleniumBase n'est pas déjà installé dans le kernel.
# %pip install seleniumbase beautifulsoup4 pandas --upgrade

## Imports

In [11]:
import asyncio
import json
import re
from datetime import datetime
from pathlib import Path
from urllib.parse import parse_qs, urlparse

import pandas as pd
from bs4 import BeautifulSoup
from seleniumbase import cdp_driver

## Configuration

In [12]:
# Fonctionne que le kernel démarre à la racine du dépôt ou dans notebooks/.
WORKING_DIR = Path.cwd().resolve()
NOTEBOOK_DIR = WORKING_DIR / "notebooks" if (WORKING_DIR / "notebooks").is_dir() else WORKING_DIR

CSV_PATTERN = "indeed_offres_*.csv"
CSV_CANDIDATES = sorted(NOTEBOOK_DIR.glob(CSV_PATTERN))
if not CSV_CANDIDATES:
    raise FileNotFoundError(
        f"Aucun fichier {CSV_PATTERN!r} trouvé dans {NOTEBOOK_DIR}."
    )
CSV_PATH = CSV_CANDIDATES[0]
OUTPUT_DIR = NOTEBOOK_DIR
DEBUG_SCREENSHOT = NOTEBOOK_DIR / "indeed_single_offer_debug.png"
MAX_LOAD_ATTEMPTS = 6

SOURCE_COLUMNS = [
    "titre",
    "entreprise",
    "lieu",
    "salaire",
    "date",
    "description",
    "badges",
    "sponsorise",
    "job_key",
    "lien",
]

# Seules ces informations ne sont pas réextraites de la page détaillée.
LEGACY_COLUMNS = [
    "badges",
    "sponsorise",
]

## Sélection de la première offre du premier CSV

In [13]:
def extract_indeed_job_key(url):
    if not url:
        return None

    parsed = urlparse(str(url).strip())
    query_key = parse_qs(parsed.query).get("jk", [None])[0]
    if query_key:
        return query_key.strip()

    match = re.search(r"(?:[?&]jk=|/viewjob/)([A-Za-z0-9_-]+)", str(url))
    return match.group(1) if match else None


def validate_indeed_url(url):
    parsed = urlparse(str(url).strip())
    hostname = (parsed.hostname or "").lower()
    is_indeed_host = (
        hostname == "indeed.com"
        or hostname.startswith("indeed.")
        or ".indeed." in hostname
    )
    if parsed.scheme not in {"http", "https"} or not is_indeed_host:
        raise ValueError(f"URL Indeed invalide : {url!r}")
    return str(url).strip()


def load_first_offer(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.is_file():
        raise FileNotFoundError(f"CSV source introuvable : {csv_path}")

    source_df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)
    missing_columns = [column for column in SOURCE_COLUMNS if column not in source_df.columns]
    if missing_columns:
        raise ValueError(f"Colonnes historiques manquantes : {missing_columns}")

    if source_df.empty:
        raise ValueError(f"Le CSV source ne contient aucune offre : {csv_path}")

    row = source_df.iloc[0][SOURCE_COLUMNS].copy()
    selected_job_key = str(row["job_key"]).strip()
    if not selected_job_key:
        raise ValueError("La première ligne du CSV ne contient pas de job_key.")
    offer_url = validate_indeed_url(row["lien"])
    url_job_key = extract_indeed_job_key(offer_url)
    if url_job_key != selected_job_key:
        raise ValueError(
            f"Le job_key de la ligne ({selected_job_key}) ne correspond pas à celui de l'URL ({url_job_key})."
        )

    return row, offer_url, selected_job_key


source_row, offer_url, selected_job_key = load_first_offer(CSV_PATH)
print(f"CSV sélectionné : {CSV_PATH}")
print("Ligne sélectionnée : 1")
print(f"Offre sélectionnée : {source_row['titre']}")
print(f"URL : {offer_url}")

CSV sélectionné : /Users/gauthier/Developer/Actif/cv-job-matcher/notebooks/indeed_offres_20260818_141028.csv
Ligne sélectionnée : 1
Offre sélectionnée : Mission Collaborateur.trice Compliance en banque - 3 mois
URL : https://ch-fr.indeed.com/viewjob?jk=2eb4f28fcc40621a


## Chargement de la page Indeed

In [14]:
CAPTCHA_MARKERS = [
    "captcha",
    "cf-turnstile",
    "just a moment",
    "verify you are human",
    "vérifiez que vous êtes humain",
]


def page_contains_offer(html):
    soup = BeautifulSoup(html, "html.parser")
    has_title = soup.select_one("[data-testid='jobsearch-JobInfoHeader-title'], h1") is not None
    has_description = soup.select_one("#jobDescriptionText") is not None
    has_job_posting = any(
        "JobPosting" in (script.string or script.get_text())
        for script in soup.select("script[type='application/ld+json']")
    )
    return has_description and (has_title or has_job_posting)


async def scrape_single_indeed_offer(url):
    driver = await cdp_driver.start_async(headless=False, xvfb=True)
    page = None

    try:
        print(f"Navigation vers : {url}")
        page = await driver.get(url)

        for attempt in range(1, MAX_LOAD_ATTEMPTS + 1):
            await asyncio.sleep(2 if attempt == 1 else 1)
            html = await page.get_content()

            if page_contains_offer(html):
                print(f"Page détaillée chargée (tentative {attempt}/{MAX_LOAD_ATTEMPTS}).")
                return html

            page_lower = html.lower()
            if any(marker in page_lower for marker in CAPTCHA_MARKERS):
                print(f"CAPTCHA détecté (tentative {attempt}/{MAX_LOAD_ATTEMPTS}).")
                await page.solve_captcha()
                await asyncio.sleep(3)
            else:
                print(f"Attente du titre et de la description ({attempt}/{MAX_LOAD_ATTEMPTS})...")

        current_url = await page.get_current_url()
        await page.save_screenshot(filename=str(DEBUG_SCREENSHOT), full_page=True)
        raise RuntimeError(
            "La page Indeed n'a pas fourni de titre et de description après "
            f"{MAX_LOAD_ATTEMPTS} tentatives. URL finale : {current_url}. "
            f"Capture : {DEBUG_SCREENSHOT}"
        )
    except Exception:
        if page is not None and not DEBUG_SCREENSHOT.exists():
            try:
                await page.save_screenshot(filename=str(DEBUG_SCREENSHOT), full_page=True)
            except Exception:
                pass
        raise
    finally:
        driver.stop()

In [15]:
# Jupyter autorise await directement au niveau de la cellule.
html = await scrape_single_indeed_offer(offer_url)
print(f"DOM rendu récupéré : {len(html):,} caractères")

Navigation vers : https://ch-fr.indeed.com/viewjob?jk=2eb4f28fcc40621a
CAPTCHA détecté (tentative 1/6).
Page détaillée chargée (tentative 2/6).
DOM rendu récupéré : 603,132 caractères


## Extraction JSON-LD et fallbacks DOM

In [16]:
def normalize_inline(value):
    if value is None:
        return None
    text = re.sub(r"\s+", " ", str(value).replace("\xa0", " ")).strip()
    return text or None


def html_to_plain_text(value):
    if not value:
        return None
    soup = BeautifulSoup(str(value), "html.parser")
    lines = [normalize_inline(line) for line in soup.get_text("\n").splitlines()]
    return "\n".join(line for line in lines if line) or None


def iter_json_objects(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from iter_json_objects(child)
    elif isinstance(value, list):
        for child in value:
            yield from iter_json_objects(child)


def find_job_posting(soup):
    for script in soup.select("script[type='application/ld+json']"):
        payload = script.string or script.get_text()
        try:
            decoded = json.loads(payload)
        except (TypeError, json.JSONDecodeError):
            continue

        for candidate in iter_json_objects(decoded):
            object_types = candidate.get("@type", [])
            if isinstance(object_types, str):
                object_types = [object_types]
            if "JobPosting" in object_types:
                return candidate
    return None


def first_mapping(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, list):
        return next((item for item in value if isinstance(item, dict)), {})
    return {}


def serialize_scalar_list(values):
    normalized = [normalize_inline(value) for value in values]
    return " | ".join(value for value in normalized if value) or None


def sanitize_column_part(value):
    cleaned = re.sub(r"[^0-9A-Za-z]+", "_", str(value)).strip("_").lower()
    return cleaned or "value"


def flatten_jsonld(value, path=()):
    flattened = {}

    if isinstance(value, dict):
        for key, child in value.items():
            flattened.update(flatten_jsonld(child, (*path, sanitize_column_part(key))))
    elif isinstance(value, list):
        if all(not isinstance(item, (dict, list)) for item in value):
            flattened["detail_jsonld_" + "_".join(path)] = serialize_scalar_list(value)
        else:
            for index, child in enumerate(value):
                flattened.update(flatten_jsonld(child, (*path, str(index))))
    else:
        column = "detail_jsonld_" + "_".join(path)
        if path and path[-1] == "description":
            flattened[column] = html_to_plain_text(value)
        elif isinstance(value, bool):
            flattened[column] = value
        else:
            flattened[column] = normalize_inline(value)

    return flattened


REDUNDANT_JSONLD_PATHS = {
    "context",
    "type",
    "dateposted",
    "description",
    "directapply",
    "employmenttype",
    "title",
    "url",
    "validthrough",
    "hiringorganization_type",
    "hiringorganization_logo",
    "hiringorganization_name",
    "hiringorganization_sameas",
    "joblocation_type",
    "joblocation_address_type",
    "joblocation_address_addresscountry",
    "joblocation_address_addresslocality",
    "joblocation_address_addressregion",
}


def remove_redundant_jsonld_columns(flattened):
    unique_columns = {}
    prefix = "detail_jsonld_"
    for column, value in flattened.items():
        path = column.removeprefix(prefix)
        # Les index de listes ne changent pas la signification du chemin JSON-LD.
        normalized_path = "_".join(part for part in path.split("_") if not part.isdigit())
        if normalized_path not in REDUNDANT_JSONLD_PATHS:
            unique_columns[column] = value
    return unique_columns


def first_text(soup, selectors):
    for selector in selectors:
        element = soup.select_one(selector)
        if element:
            text = normalize_inline(element.get_text(" ", strip=True))
            if text:
                return text
    return None


def extract_company_metrics(soup):
    container = soup.select_one("[data-testid='jobsearch-CompanyInfoContainer']")
    header_text = normalize_inline(container.get_text(" ", strip=True)) if container else ""

    rating_match = re.search(r"(\d+(?:[.,]\d+)?)\s*/\s*5", header_text or "")
    rating = rating_match.group(1).replace(",", ".") if rating_match else None

    reviews_text = first_text(soup, ["[data-testid='companyReviewLink']", "a[href*='/reviews']"])
    reviews_match = re.search(r"([\d\s.,]+)\s+(?:avis|reviews?)\b", reviews_text or "", re.I)
    reviews_count = re.sub(r"\D", "", reviews_match.group(1)) if reviews_match else None
    return rating, reviews_count


def extract_offer_details(html, fallback_url):
    soup = BeautifulSoup(html, "html.parser")
    job_posting = find_job_posting(soup) or {}

    organization = first_mapping(job_posting.get("hiringOrganization"))
    job_location = first_mapping(job_posting.get("jobLocation"))
    address = first_mapping(job_location.get("address"))

    canonical_element = soup.select_one("link[rel='canonical']")
    canonical_url = validate_indeed_url(
        canonical_element.get("href") if canonical_element else fallback_url
    )

    description_text = html_to_plain_text(job_posting.get("description"))
    if not description_text:
        description_element = soup.select_one("#jobDescriptionText")
        description_text = (
            html_to_plain_text(str(description_element)) if description_element else None
        )

    location = first_text(
        soup,
        [
            "[data-testid='inlineHeader-companyLocation']",
            "[data-testid='jobLocationText']",
            "#jobLocationText",
            ".jobsearch-JobInfoHeader-companyLocation",
        ],
    )
    if not location:
        location = serialize_scalar_list(
            [address.get("addressLocality"), address.get("addressRegion"), address.get("addressCountry")]
        )

    employment_type = job_posting.get("employmentType")
    if isinstance(employment_type, list):
        employment_type = serialize_scalar_list(employment_type)
    else:
        employment_type = normalize_inline(employment_type)

    rating, reviews_count = extract_company_metrics(soup)
    details = {
        "detail_title": normalize_inline(job_posting.get("title"))
        or first_text(soup, ["[data-testid='jobsearch-JobInfoHeader-title']", "h1"]),
        "detail_company": normalize_inline(organization.get("name"))
        or first_text(soup, ["[data-testid='inlineHeader-companyName']", "[data-company-name='true']"]),
        "detail_company_url": normalize_inline(organization.get("sameAs")),
        "detail_company_logo": normalize_inline(organization.get("logo")),
        "detail_canonical_url": canonical_url,
        "detail_job_key": extract_indeed_job_key(canonical_url),
        "detail_location": location,
        "detail_address_country": normalize_inline(address.get("addressCountry")),
        "detail_address_locality": normalize_inline(address.get("addressLocality")),
        "detail_address_region": normalize_inline(address.get("addressRegion")),
        "detail_date_posted": normalize_inline(job_posting.get("datePosted")),
        "detail_valid_through": normalize_inline(job_posting.get("validThrough")),
        "detail_employment_type": employment_type,
        "detail_direct_apply": job_posting.get("directApply"),
        "detail_salary_job_type": first_text(
            soup,
            ["[data-testid='salaryInfoAndJobType']", "#salaryInfoAndJobType"],
        ),
        "detail_description_text": description_text,
        "detail_company_rating": rating,
        "detail_company_reviews_count": reviews_count,
    }

    # Seules les informations JSON-LD qui ne sont pas déjà dans detail_* sont ajoutées.
    jsonld_columns = remove_redundant_jsonld_columns(flatten_jsonld(job_posting))
    details.update(jsonld_columns)
    return details, bool(job_posting)

## Fusion et contrôle de cohérence

In [17]:
details, jsonld_found = extract_offer_details(html, offer_url)

if details["detail_job_key"] != selected_job_key:
    raise ValueError(
        "Refus de fusionner : le job_key canonique "
        f"{details['detail_job_key']!r} diffère du job_key sélectionné {selected_job_key!r}."
    )
if not details["detail_title"]:
    raise ValueError("Le titre détaillé est absent de la page Indeed.")
if not details["detail_description_text"]:
    raise ValueError("La description détaillée est absente de la page Indeed.")

enriched_record = {column: source_row[column] for column in LEGACY_COLUMNS}
enriched_record.update(details)
detail_columns = [column for column in enriched_record if column not in LEGACY_COLUMNS]
enriched_df = pd.DataFrame(
    [enriched_record],
    columns=[*LEGACY_COLUMNS, *detail_columns],
)

print(f"JSON-LD JobPosting détecté : {jsonld_found}")
print(f"Colonnes héritées conservées : {len(LEGACY_COLUMNS)}")
print(f"Colonnes détaillées : {len(detail_columns)}")
display(enriched_df.T.rename(columns={0: "valeur"}))

JSON-LD JobPosting détecté : True
Colonnes héritées conservées : 2
Colonnes détaillées : 18


,valeur
badges,N/A
sponsorise,Non
detail_title,Mission Collaborateur.trice Compliance en banq...
detail_company,Academic Work
detail_company_url,https://ch-fr.indeed.com/cmp/Academic-Work
detail_company_logo,https://d2q79iu7y748jz.cloudfront.net/s/_squar...
detail_canonical_url,https://ch-fr.indeed.com/viewjob?jk=2eb4f28fcc...
detail_job_key,2eb4f28fcc40621a
detail_location,"Genève, GE"
detail_address_country,CH


## Export CSV

In [18]:
if len(enriched_df) != 1:
    raise AssertionError(f"Une seule ligne était attendue, obtenu : {len(enriched_df)}")
if enriched_df.columns[: len(LEGACY_COLUMNS)].tolist() != LEGACY_COLUMNS:
    raise AssertionError("Les colonnes héritées conservées ne sont pas en tête du CSV.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = OUTPUT_DIR / f"indeed_offre_detail_{selected_job_key}_{timestamp}.csv"
enriched_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"CSV enrichi sauvegardé : {output_path}")
print(f"Dimensions : {enriched_df.shape[0]} ligne × {enriched_df.shape[1]} colonnes")

CSV enrichi sauvegardé : /Users/gauthier/Developer/Actif/cv-job-matcher/notebooks/indeed_offre_detail_2eb4f28fcc40621a_20260818_150351.csv
Dimensions : 1 ligne × 20 colonnes
